In [1]:
import os, json, time, urllib.request, zipfile
import numpy as np
import cv2
from tqdm import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import pandas as pd
from IPython.display import display

In [2]:
COCO_VAL_IMAGES      = "coco/images/val2017"          # folder with val images
COCO_ANNOTATIONS     = "coco/annotations/instances_val2017.json"
MODEL_PATH           = "efficientdet_lite4.tflite"
RESULTS_JSON         = "efficientdet_lite4_coco_results.json"
THRESHOLD            = 0.001   # low threshold → let COCOeval filter
INPUT_SIZE           = 640

In [4]:

# ─────────────────────────────────────────────────────────────────
# STEP 1 — Model path (manual download, no auto-download)
# ─────────────────────────────────────────────────────────────────
MODEL_PATH = "efficientdet_lite4.tflite"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        "❌ Model not found!\n"
        "Please download it manually from:\n"
        "https://tfhub.dev/tensorflow/lite-model/efficientdet/lite4/detection/metadata/2?lite-format=tflite\n"
        f"And save it as: {os.path.abspath(MODEL_PATH)}"
    )

print(f"✅ Model found: {MODEL_PATH}")

✅ Model found: efficientdet_lite4.tflite


In [5]:
# ─────────────────────────────────────────────────────────────────
# STEP 2 — Download COCO val2017 if needed
# ─────────────────────────────────────────────────────────────────
def download_coco_val():
    os.makedirs("coco/images", exist_ok=True)
    os.makedirs("coco/annotations", exist_ok=True)

    if not os.path.exists(COCO_VAL_IMAGES):
        print("Downloading COCO val2017 images (1GB)...")
        urllib.request.urlretrieve(
            "http://images.cocodataset.org/zips/val2017.zip",
            "val2017.zip"
        )
        with zipfile.ZipFile("val2017.zip") as z:
            z.extractall("coco/images/")
        os.remove("val2017.zip")
        print("✅ Images downloaded")

    if not os.path.exists(COCO_ANNOTATIONS):
        print("Downloading COCO annotations...")
        urllib.request.urlretrieve(
            "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
            "annotations.zip"
        )
        with zipfile.ZipFile("annotations.zip") as z:
            z.extractall("coco/")
        os.remove("annotations.zip")
        print("✅ Annotations downloaded")

download_coco_val()

In [6]:
# ─────────────────────────────────────────────────────────────────
# STEP 3 — Load COCO ground truth
# ─────────────────────────────────────────────────────────────────
coco_gt   = COCO(COCO_ANNOTATIONS)
image_ids = sorted(coco_gt.getImgIds())

# COCO category id → name mapping
cat_id_to_name = {cat["id"]: cat["name"] for cat in coco_gt.loadCats(coco_gt.getCatIds())}

# EfficientDet output class index → COCO category id
# EfficientDet-Lite4 uses 1-indexed COCO labels
# index 0 = background, index 1 = person (coco id=1), etc.
# We build a mapping from the COCO category list (sorted by id)
coco_cat_ids_sorted = sorted(coco_gt.getCatIds())  # 80 categories
efficientdet_to_coco = {i+1: coco_cat_ids_sorted[i] for i in range(len(coco_cat_ids_sorted))}

print(f"✅ COCO loaded: {len(image_ids)} images, {len(cat_id_to_name)} categories")

loading annotations into memory...
Done (t=0.39s)
creating index...
index created!
✅ COCO loaded: 5000 images, 80 categories


In [7]:

# ─────────────────────────────────────────────────────────────────
# STEP 4 — Load TFLite model
# ─────────────────────────────────────────────────────────────────
# STEP 4 — Load TFLite model (Windows fix)
# ─────────────────────────────────────────────────────────────────
from tensorflow.lite.python.interpreter import Interpreter

interpreter = Interpreter(model_path=MODEL_PATH, num_threads=4)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print(f"✅ Model loaded | Input: {input_details[0]['shape']}")

✅ Model loaded | Input: [  1 640 640   3]


c:\Users\admin\PFE\PFE-Obstacle-Detection\pfe\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [8]:
# ─────────────────────────────────────────────────────────────────
# STEP 5 — Run inference on COCO val2017
# ─────────────────────────────────────────────────────────────────
results = []
inference_times = []

print(f"\n🚀 Running EfficientDet-Lite4 on {len(image_ids)} COCO val2017 images...")

for img_id in tqdm(image_ids):
    img_info = coco_gt.loadImgs(img_id)[0]
    img_path = os.path.join(COCO_VAL_IMAGES, img_info["file_name"])

    if not os.path.exists(img_path):
        continue

    # Load & preprocess
    image = cv2.imread(img_path)
    if image is None:
        continue

    orig_h, orig_w = image.shape[:2]
    image_rgb     = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_resized = cv2.resize(image_rgb, (INPUT_SIZE, INPUT_SIZE))
    input_tensor  = np.expand_dims(image_resized, axis=0).astype(np.uint8)

    # Inference
    interpreter.set_tensor(input_details[0]['index'], input_tensor)
    t0 = time.time()
    interpreter.invoke()
    inference_times.append((time.time() - t0) * 1000)

    # Outputs: boxes [ymin,xmin,ymax,xmax] normalized, classes, scores, count
    boxes   = interpreter.get_tensor(output_details[0]['index'])[0]
    classes = interpreter.get_tensor(output_details[1]['index'])[0]
    scores  = interpreter.get_tensor(output_details[2]['index'])[0]
    count   = int(interpreter.get_tensor(output_details[3]['index'])[0])

    for i in range(count):
        score    = float(scores[i])
        if score < THRESHOLD:
            continue

        class_idx = int(classes[i]) + 1   # EfficientDet is 0-indexed internally
        coco_cat  = efficientdet_to_coco.get(class_idx, -1)
        if coco_cat == -1:
            continue

        ymin, xmin, ymax, xmax = boxes[i]

        # Convert to COCO format: [x, y, width, height] in pixels
        x      = float(xmin * orig_w)
        y      = float(ymin * orig_h)
        width  = float((xmax - xmin) * orig_w)
        height = float((ymax - ymin) * orig_h)

        results.append({
            "image_id":    img_id,
            "category_id": coco_cat,
            "bbox":        [x, y, width, height],
            "score":       score,
        })

# Save results
with open(RESULTS_JSON, "w") as f:
    json.dump(results, f)

avg_ms = np.mean(inference_times)
print(f"\n✅ Done | {len(results)} detections | Avg inference: {avg_ms:.1f} ms/image")


🚀 Running EfficientDet-Lite4 on 5000 COCO val2017 images...


100%|██████████| 5000/5000 [08:20<00:00,  9.99it/s]



✅ Done | 114274 detections | Avg inference: 94.4 ms/image


In [9]:
# ─────────────────────────────────────────────────────────────────
# STEP 6 — COCO Evaluation (mAP50, mAP50-95)
# ─────────────────────────────────────────────────────────────────
print("\n📊 Running COCO evaluation...")

coco_dt  = coco_gt.loadRes(RESULTS_JSON)
coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

map50_95 = coco_eval.stats[0]   # mAP @ IoU 0.50:0.95
map50    = coco_eval.stats[1]   # mAP @ IoU 0.50


📊 Running COCO evaluation...
Loading and preparing results...
DONE (t=0.80s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=8.38s).
Accumulating evaluation results...
DONE (t=1.66s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.063
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.091
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.066
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.036
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.065
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.091
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.047
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.074
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.075
 Average Recall     (AR) @[ IoU=0.50:0

In [10]:
# ─────────────────────────────────────────────────────────────────
# STEP 7 — Per-class AP table (like Table 1 in DeepNAVI)
# ─────────────────────────────────────────────────────────────────
print("\n📋 Computing per-class AP...")

rows = []
for cat_id in coco_cat_ids_sorted:
    cat_name  = cat_id_to_name[cat_id]
    coco_eval.params.catIds = [cat_id]
    coco_eval.evaluate()
    coco_eval.accumulate()

    ap50_95 = coco_eval.stats[0]
    ap50    = coco_eval.stats[1]

    rows.append({
        "Class":      cat_name.capitalize(),
        "mAP50":      round(float(ap50),    3),
        "mAP50-95":   round(float(ap50_95), 3),
        "Accuracy":   f"{round(float(ap50_95) * 100, 1)}%",
    })

df = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False).reset_index(drop=True)

# Add medals
medals = {0: "🥇", 1: "🥈", 2: "🥉"}
df.insert(0, "Rank", [f"{medals.get(i, '')} {i+1}" for i in range(len(df))])

# Add average row
avg_row = pd.DataFrame([{
    "Rank":     "—",
    "Class":    "Average",
    "mAP50":    round(map50, 3),
    "mAP50-95": round(map50_95, 3),
    "Accuracy": f"{round(map50_95 * 100, 1)}%",
}])

df_final = pd.concat([df, avg_row], ignore_index=True)

# Display styled table
styled = (
    df_final.style
    .set_caption("EfficientDet-Lite4 — Per-class results on COCO val2017")
    .set_properties(**{"text-align": "left"})
    .set_properties(**{"font-weight": "bold"}, subset=["Accuracy"])
    .apply(lambda x: ["font-style: italic; border-top: 2px solid black"] * len(x)
           if x.name == len(df_final) - 1 else [""] * len(x), axis=1)
    .set_table_styles([
        {"selector": "thead th", "props": [("border-bottom", "2px solid black"), ("padding", "8px 14px")]},
        {"selector": "td", "props": [("padding", "6px 14px"), ("border-bottom", "1px solid #ddd")]},
        {"selector": "table", "props": [("border-collapse", "collapse"), ("font-size", "13px")]},
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "14px"), ("text-align", "left"), ("margin-bottom", "8px")]},
    ])
    .hide(axis="index")
)

display(styled)

# Also save to CSV
df_final.to_csv("efficientdet_lite4_per_class.csv", index=False)
print(f"\n✅ Saved to efficientdet_lite4_per_class.csv")
print(f"\n🏆 Overall mAP50:    {map50:.3f}")
print(f"🏆 Overall mAP50-95: {map50_95:.3f}")
print(f"⏱  Avg inference:    {avg_ms:.1f} ms/image")


📋 Computing per-class AP...
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=2.56s).
Accumulating evaluation results...
DONE (t=0.34s).
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.13s).
Accumulating evaluation results...
DONE (t=0.02s).
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.35s).
Accumulating evaluation results...
DONE (t=0.06s).
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.10s).
Accumulating evaluation results...
DONE (t=0.01s).
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.07s).
Accumulating evaluation results...
DONE (t=0.01s).
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.07s).
Accumulating evaluation results...
DONE (t=0.02s).
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.07s).
Accumulating evaluation results...
DONE (t=0.02s).
Running per image evaluation...
Evaluate annotation ty

Rank,Class,mAP50,mAP50-95,Accuracy
🥇 1,Person,0.091000,0.063000,6.3%
🥈 2,Bicycle,0.091000,0.063000,6.3%
🥉 3,Car,0.091000,0.063000,6.3%
4,Motorcycle,0.091000,0.063000,6.3%
5,Airplane,0.091000,0.063000,6.3%
6,Bus,0.091000,0.063000,6.3%
7,Train,0.091000,0.063000,6.3%
8,Truck,0.091000,0.063000,6.3%
9,Boat,0.091000,0.063000,6.3%
10,Traffic light,0.091000,0.063000,6.3%



✅ Saved to efficientdet_lite4_per_class.csv

🏆 Overall mAP50:    0.091
🏆 Overall mAP50-95: 0.063
⏱  Avg inference:    94.4 ms/image
